In [ ]:
# Google Colab-only setup — run this notebook in its own fresh Colab runtime.
import sys
if "google.colab" not in sys.modules:
    raise RuntimeError(
        "This session 4 is Google Colab-only. Open https://colab.research.google.com/, "
        "upload this notebook, and run it there."
    )

%pip install -q ultralytics==8.4.102 numpy pandas matplotlib

import torch
torch.manual_seed(0)


In [ ]:
# Independent Colab assets — this notebook never reads another session's files.
from pathlib import Path
import os

SESSION_WORKSPACE = Path("/content/yolo_object_detection_session_04")
SESSION_WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(SESSION_WORKSPACE)

from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

SESSION_DATASETS = SESSION_WORKSPACE / "datasets"
settings.update({"datasets_dir": str(SESSION_DATASETS)})
COCO128_INFO = check_det_dataset("coco128.yaml", autodownload=True)
COCO128_YAML = Path(COCO128_INFO.get("yaml_file", "coco128.yaml"))
BASELINE_MODEL = YOLO("yolo11n.pt")  # Downloads and caches this notebook's pretrained weights.
print(f"Colab-only session 4: workspace={SESSION_WORKSPACE} | dataset={COCO128_YAML} | model=yolo11n.pt")


## 1. Fix the evaluation settings

A metric without its model, split, evaluator, image size, and IoU convention is incomplete. This block records those choices before evaluation.

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import csv
import json
import os
import random
import tempfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_CHECKPOINT = "yolo11n.pt"
IMAGE_SIZE = 416
EVALUATION_IOU = 0.50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    INSTALLED_ULTRALYTICS = version("ultralytics")
except PackageNotFoundError:
    INSTALLED_ULTRALYTICS = "not installed"

IN_COLAB = "google.colab" in sys.modules
RUN_REAL_COCO128_VALIDATION = IN_COLAB

print({
    "seed": SEED, "model": MODEL_CHECKPOINT, "ultralytics": INSTALLED_ULTRALYTICS,
    "device": DEVICE, "imgsz": IMAGE_SIZE, "evaluation_iou": EVALUATION_IOU,
    "real_coco128_enabled": RUN_REAL_COCO128_VALIDATION
})

## 2. one-to-one matching with reasons

Predictions are processed from highest to lowest confidence. A TP must have the same class, reach IoU 0.50, and claim a truth that no earlier prediction claimed. Other predictions receive an explicit reason: duplicate, localization, class confusion, or background.

In [ ]:
truths = [
    {"id": "gt-person", "class": "person", "box": [0, 0, 100, 100]},
    {"id": "gt-dog", "class": "dog", "box": [150, 0, 250, 100]},
    {"id": "gt-bicycle", "class": "bicycle", "box": [0, 150, 100, 250]},
]
predictions = [
    {"id": "p1", "class": "person", "score": 0.95, "box": [0, 0, 100, 100]},
    {"id": "p2", "class": "person", "score": 0.90, "box": [5, 5, 95, 95]},
    {"id": "p3", "class": "dog", "score": 0.85, "box": [160, 10, 245, 95]},
    {"id": "p4", "class": "bicycle", "score": 0.80, "box": [30, 180, 130, 280]},
    {"id": "p5", "class": "cat", "score": 0.75, "box": [0, 150, 100, 250]},
    {"id": "p6", "class": "bicycle", "score": 0.70, "box": [0, 150, 100, 250]},
]

In [ ]:
# TODO: Complete IoU and score-sorted one-to-one matching with explicit FP reasons.
# HINT: Search only unmatched same-class truths for a TP; inspect all truths to explain an FP.
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = intersection_width * intersection_height
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def match_predictions(predictions, truths, iou_threshold=0.50):
    matched_truth_ids = set()
    rows = []
    for prediction in sorted(predictions, key=lambda item: item["score"], reverse=True):
        same_class = [truth for truth in truths if truth["class"] == prediction["class"]]
        unmatched = [truth for truth in same_class if truth["id"] not in matched_truth_ids]
        best_unmatched = max(unmatched, key=lambda truth: box_iou(prediction["box"], truth["box"]), default=None)
        best_unmatched_iou = box_iou(prediction["box"], best_unmatched["box"]) if best_unmatched else 0.0

        if best_unmatched is not None and best_unmatched_iou >= iou_threshold:
            matched_truth_ids.add(best_unmatched["id"])
            outcome, reason, matched_id = "TP", "same class + IoU threshold + unclaimed truth", best_unmatched["id"]
            reported_iou = best_unmatched_iou
        else:
            best_same = max(((box_iou(prediction["box"], truth["box"]), truth) for truth in same_class), default=(0.0, None), key=lambda pair: pair[0])
            best_any = max(((box_iou(prediction["box"], truth["box"]), truth) for truth in truths), default=(0.0, None), key=lambda pair: pair[0])
            if best_same[1] is not None and best_same[0] >= iou_threshold and best_same[1]["id"] in matched_truth_ids:
                reason = "duplicate: qualifying truth already claimed"
            elif best_same[1] is not None and best_same[0] > 0:
                reason = "localization: correct class but IoU below threshold"
            elif best_any[1] is not None and best_any[0] >= iou_threshold:
                reason = "class confusion: overlaps a truth of another class"
            else:
                reason = "background: no qualifying overlap"
            outcome, matched_id, reported_iou = "FP", None, max(best_same[0], best_any[0])
        rows.append({
            "prediction": prediction["id"], "class": prediction["class"], "score": prediction["score"],
            "outcome": outcome, "best_iou": reported_iou, "matched_truth": matched_id, "reason": reason
        })
    return pd.DataFrame(rows), matched_truth_ids

match_table, matched_truth_ids = match_predictions(predictions, truths, EVALUATION_IOU)

In [ ]:
display(match_table.round({"best_iou": 3}))
assert len(matched_truth_ids) == len(set(matched_truth_ids)), "A truth was matched twice."
assert match_table["outcome"].tolist() == ["TP", "FP", "TP", "FP", "FP", "TP"]
reason_text = " ".join(match_table.loc[match_table.outcome == "FP", "reason"].tolist())
for expected_reason in ("duplicate", "localization", "class confusion"):
    assert expected_reason in reason_text
print("Matched truth IDs:", sorted(matched_truth_ids))

## 3. Compute point precision and recall

Precision divides by reported detections (`TP + FP`): *Of the boxes reported, how many were correct?* Recall divides by ground-truth objects (`TP + FN`): *Of the objects labeled, how many were recovered?*

In [ ]:
tp = int((match_table.outcome == "TP").sum())
fp = int((match_table.outcome == "FP").sum())
fn = len(truths) - len(matched_truth_ids)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
print(f"TP={tp}, FP={fp}, FN={fn}")
print(f"precision = TP/(TP+FP) = {tp}/{tp+fp} = {precision:.3f}")
print(f"recall    = TP/(TP+FN) = {tp}/{tp+fn} = {recall:.3f}")
assert (tp, fp, fn) == (3, 3, 0) and precision == 0.5 and recall == 1.0

## 4.cumulative PR table and envelope

Use the module's descending-score outcomes `[TP, FP, TP, TP]` with three ground-truth objects. Raw precision can zig-zag; the interpolated envelope at a recall level is the best precision available at that recall or any larger recall.

In [ ]:
# TODO: Build cumulative TP/FP, precision, recall, and the right-to-left precision envelope; then plot both.
# HINT: recall belongs on x, precision on y; use np.maximum.accumulate on reversed precision.
ranked_outcomes = np.array(["TP", "FP", "TP", "TP"])
number_of_truths = 3
cumulative_tp = np.cumsum(ranked_outcomes == "TP")
cumulative_fp = np.cumsum(ranked_outcomes == "FP")
rank_precision = cumulative_tp / (cumulative_tp + cumulative_fp)
rank_recall = cumulative_tp / number_of_truths
precision_envelope = np.maximum.accumulate(rank_precision[::-1])[::-1]
pr_table = pd.DataFrame({
    "rank": np.arange(1, len(ranked_outcomes)+1), "outcome": ranked_outcomes,
    "cumulative_tp": cumulative_tp, "cumulative_fp": cumulative_fp,
    "precision": rank_precision, "recall": rank_recall, "interpolated_precision": precision_envelope
})

fig, axis = plt.subplots(figsize=(6, 4))
axis.plot(rank_recall, rank_precision, "o--", label="raw ranked points")
axis.step(rank_recall, precision_envelope, where="post", linewidth=2.5, label="interpolated envelope")
axis.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1.03), ylim=(0, 1.03), title="Controlled precision–recall example")
axis.grid(alpha=0.25); axis.legend(); plt.tight_layout()

In [ ]:
display(pr_table.round(3))
assert np.allclose(cumulative_tp, [1, 1, 2, 3])
assert np.allclose(cumulative_fp, [0, 1, 1, 1])
assert np.isclose(rank_precision[-1], 0.75) and np.isclose(rank_recall[-1], 1.0)
print("Final point: precision=0.75, recall=1.00")
print("The plotted envelope is an interpolation aid; this short calculation does not claim to duplicate a toolkit's AP integration rule.")

## 5. Create and save the 96/32 teaching split

A fixed manifest prevents accidental image overlap. The default filenames form a deterministic stand-in for offline validation; when real validation is enabled, the same seed-42 logic is applied to actual COCO128 paths. Each CSV explicitly pairs an image with its same-stem label.

In [ ]:
def split_96_32(items, seed=42):
    assert len(items) == 128, f"Expected 128 items, received {len(items)}"
    shuffled = list(items)
    random.Random(seed).shuffle(shuffled)
    return sorted(shuffled[:96]), sorted(shuffled[96:])

def label_for_image(image_name):
    return str(Path(image_name).with_suffix(".txt")).replace("/images/", "/labels/")

def save_manifest(path, image_items):
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["image", "label"])
        writer.writeheader()
        writer.writerows({"image": item, "label": label_for_image(item)} for item in image_items)

placeholder_images = [f"coco128/images/train2017/{index:012d}.jpg" for index in range(1, 129)]
train_images, validation_images = split_96_32(placeholder_images, SEED)
ARTIFACT_DIR = Path(tempfile.gettempdir()) / "yolo_session04_manifests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
train_manifest_path = ARTIFACT_DIR / "train_96.csv"
validation_manifest_path = ARTIFACT_DIR / "validation_32.csv"
save_manifest(train_manifest_path, train_images); save_manifest(validation_manifest_path, validation_images)

assert len(train_images) == 96 and len(validation_images) == 32
assert set(train_images).isdisjoint(validation_images)
assert all(Path(row).stem == Path(label_for_image(row)).stem for row in train_images + validation_images)
print("Saved:", train_manifest_path, "and", validation_manifest_path)
print("Counts: train=96, validation=32, overlap=0")
print("First held-out pairs:")
display(pd.read_csv(validation_manifest_path).head(3))

## 6. Validate: deterministic default or explicit real COCO128 run

The fallback report is simulated from declared counts and per-class AP values so notebook logic can be validated offline; it is not a pretrained-model measurement. The optional branch downloads/locates COCO128, rebuilds the real 96/32 manifests, and calls the pinned Ultralytics evaluator on only the held-out list.

In [ ]:
if RUN_REAL_COCO128_VALIDATION:
    if INSTALLED_ULTRALYTICS != "8.4.102":
        raise RuntimeError(f"Real validation requires ultralytics==8.4.102, found {INSTALLED_ULTRALYTICS}")
    from ultralytics import YOLO
    from ultralytics.data.utils import check_det_dataset
    dataset_info = check_det_dataset("coco128.yaml", autodownload=True)
    real_image_root = Path(dataset_info["train"])
    real_images = sorted(str(path.resolve()) for path in real_image_root.rglob("*.jpg"))
    if len(real_images) != 128:
        raise RuntimeError(f"Expected exactly 128 COCO128 images, found {len(real_images)}")
    train_images, validation_images = split_96_32(real_images, SEED)
    save_manifest(train_manifest_path, train_images); save_manifest(validation_manifest_path, validation_images)
    train_list = ARTIFACT_DIR / "train_96.txt"
    validation_list = ARTIFACT_DIR / "validation_32.txt"
    train_list.write_text("\n".join(train_images) + "\n", encoding="utf-8")
    validation_list.write_text("\n".join(validation_images) + "\n", encoding="utf-8")
    teaching_yaml = ARTIFACT_DIR / "coco128_teaching_split.yaml"
    teaching_yaml.write_text(json.dumps({
        "path": str(Path(dataset_info["path"]).resolve()), "train": str(train_list.resolve()),
        "val": str(validation_list.resolve()), "names": dataset_info["names"]
    }, indent=2), encoding="utf-8")  # JSON is valid YAML.
    model = YOLO(MODEL_CHECKPOINT)
    metrics = model.val(data=str(teaching_yaml), split="val", imgsz=IMAGE_SIZE, device=DEVICE, plots=False, verbose=False)
    metric_record = {
        "status": "measured pretrained validation", "model": MODEL_CHECKPOINT,
        "package": f"ultralytics {INSTALLED_ULTRALYTICS}", "split": "seed-42 held-out 32/128 COCO128 images",
        "evaluator": "Ultralytics val (COCO-style AP)", "imgsz": IMAGE_SIZE,
        "precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map)
    }
    class_names = dataset_info["names"] if isinstance(dataset_info["names"], dict) else dict(enumerate(dataset_info["names"]))
    support_counts = {class_id: 0 for class_id in class_names}
    for image_name in validation_images:
        label_path = Path(label_for_image(image_name))
        for row in label_path.read_text(encoding="utf-8").splitlines() if label_path.exists() else []:
            if row.strip():
                support_counts[int(row.split()[0])] += 1
    present_support = {class_names[class_id]: count for class_id, count in support_counts.items() if count}
    absent_count = sum(count == 0 for count in support_counts.values())
    support_note = f"Held-out class support: {present_support}; absent classes: {absent_count}/{len(class_names)}."
else:
    # Controlled counts: TP=18, FP=6, FN=12. AP arrays are declared teaching values.
    fallback_tp, fallback_fp, fallback_fn = 18, 6, 12
    per_class_ap50 = np.array([0.82, 0.65, 0.57])
    per_class_ap50_95 = np.array([0.55, 0.38, 0.33])
    metric_record = {
        "status": "SIMULATED fallback — not a model measurement", "model": f"{MODEL_CHECKPOINT} (not executed)",
        "package": f"target ultralytics 8.4.102; installed {INSTALLED_ULTRALYTICS}",
        "split": "seed-42 deterministic 32-name teaching manifest",
        "evaluator": "controlled fallback evaluator v1", "imgsz": IMAGE_SIZE,
        "precision": fallback_tp/(fallback_tp+fallback_fp), "recall": fallback_tp/(fallback_tp+fallback_fn),
        "mAP50": float(per_class_ap50.mean()), "mAP50-95": float(per_class_ap50_95.mean())
    }
    support_note = "Controlled support: person=16, dog=10, bicycle=4; all other COCO classes absent."

metric_table = pd.DataFrame([metric_record])
display(metric_table.round(3))
print(support_note)
assert metric_record["mAP50"] >= metric_record["mAP50-95"]
assert len(train_images) == 96 and len(validation_images) == 32 and set(train_images).isdisjoint(validation_images)

## 7. Interpret cautiously

`mAP50` uses a localization-tolerant IoU 0.50 match. COCO-style `mAP50–95` averages AP over IoU thresholds 0.50, 0.55, …, 0.95, so loose boxes are penalized more strongly. A 32-image split is sparse: some classes have few examples and many may be absent, making class AP unstable. This teaching split must not be compared with COCO leaderboard results.

In [ ]:
print(
    f"{metric_record['model']} on the {metric_record['split']}, evaluated by {metric_record['evaluator']} "
    f"at imgsz={metric_record['imgsz']}, reported precision={metric_record['precision']:.3f}, recall={metric_record['recall']:.3f}, "
    f"mAP50={metric_record['mAP50']:.3f} (IoU 0.50), and mAP50–95={metric_record['mAP50-95']:.3f} "
    f"(IoU 0.50:0.95); status: {metric_record['status']}."
)